# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My lane: CTR / Engagement Opportunity Scoring

I am choosing the **CTR / Engagement Opportunity Scoring** lane as an SEO analyst. My provisional question is: **Which content items have enough search visibility but capture fewer clicks than a comparable position would suggest, and therefore deserve review first?** The unit of analysis will be one pseudonymized content item over the trailing 90-day window. I will use observable search and engagement signals, beginning with a transparent position- and volume-adjusted baseline, then test whether a simple model improves the ranking. This lane is useful because an SEO analyst can turn the ranked output into a finite title/meta, intent-match, on-page engagement, or monitoring queue.


In [16]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)

print(f"Starter rows: {len(df):,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")
print("Analysis grain: one content item with trailing-90-day metrics")

Starter rows: 30,000
Pseudonymized clients: 32
Analysis grain: one content item with trailing-90-day metrics


## 2. The question: decision, action, cost of a wrong call

**Decision:** which content items should an SEO analyst review first for a possible click-through-rate or engagement opportunity?

**Output:** a ranked review queue with a score, a confidence or evidence label, and reason codes such as high impressions, strong position, low CTR, or weak engagement. I will judge the ranking with `precision@K`, where `K` matches a realistic review capacity, and compare it with a simple rule baseline.

**Action:** an SEO analyst or editor can inspect the highest-ranked items and choose a proportionate action: test a title or meta description, improve intent match and snippet structure, improve on-page engagement, or monitor rather than edit when the evidence is weak.

**Cost of a wrong call:** a false positive spends limited analyst and editor time on a page that does not need attention. A false negative can leave clicks or engagement on the table. Because an edit can also damage a useful page, the output is a review priority, not an automatic instruction to change content.

**Why data or ML can help:** a fixed rule can identify obvious high-volume, low-CTR pages, so it is the required baseline. Data analysis can adjust for position and volume and reveal where several signals travel together. A simple model earns its place only if it improves top-K ranking on held-out clients and remains explainable; the model itself is not the goal.


In [17]:
# This section is a written frame; the data checks live below.
print("Decision: rank visible content items for human CTR/engagement review")
print("Metric: precision@K against an observed, leakage-safe outcome defined later")
print("Primary user: SEO analyst or editor")

Decision: rank visible content items for human CTR/engagement review
Metric: precision@K against an observed, leakage-safe outcome defined later
Primary user: SEO analyst or editor


## 3. Quick look at the data (2-3 real numbers)

The starter CSV has 30,000 content items across 32 pseudonymized clients. Using a conservative visibility screen of at least 500 impressions and an average position from 1 through 20, 12,009 items qualify for review. Of those, 9,745 (81.15%) have CTR below 0.5 percentage points. The visible-page median position is 8.2 and the median CTR is 0.21 percentage points.

These numbers make the lane worth exploring for seven weeks: there is a substantial, measurable review population, but the high rate of low CTR also warns me that a simple cutoff may be too broad. Position-adjusted comparisons, minimum-volume rules, grouped validation, and human-readable reason codes are needed before recommending edits.


In [18]:
visible = df.loc[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"].between(1, 20))
].copy()
weak_ctr = visible.loc[visible["ctr"] < 0.5]

print(f"Visible pages (>=500 impressions, position 1-20): {len(visible):,}")
print(f"Visible pages with CTR < 0.5 percentage points: {len(weak_ctr):,}")
print(f"Share of visible pages with CTR < 0.5: {100 * len(weak_ctr) / len(visible):.2f}%")
print(f"Median visible position: {visible['avg_position'].median():.1f}")
print(f"Median visible CTR: {visible['ctr'].median():.2f} percentage points")

Visible pages (>=500 impressions, position 1-20): 12,009
Visible pages with CTR < 0.5 percentage points: 9,745
Share of visible pages with CTR < 0.5: 81.15%
Median visible position: 8.2
Median visible CTR: 0.21 percentage points


## 4. Careful words: what I can and can't claim

I can report **observed** relationships in this starter slice: which content items have visibility, position, CTR, and engagement patterns; whether a position-adjusted ranking concentrates low-CTR observations near the top; and whether a held-out-client test supports better decision-support than the baseline. I can make **directional** recommendations for which pages deserve human review first.

I cannot claim that low CTR proves a title or meta description is the cause, that a refresh will cause recovery, or that the analysis predicts Google rankings or algorithm factors. The starter `is_declining_label` is derived from the current `trend_direction`, so I will not use `trend_direction`, `trend_pct`, or pseudonymous IDs as ordinary model features. Any future target must be measured after a clearly separated decision point; otherwise this remains an observational ranking exercise.


In [19]:
excluded_from_features = {"content_id", "client_id", "trend_direction", "trend_pct"}
missing_columns = excluded_from_features.difference(df.columns)

assert not missing_columns, f"Expected columns are missing: {missing_columns}"
assert "is_declining_label" not in df.columns
assert len(df) == 30_000
assert df["client_id"].nunique() == 32
assert len(visible) == 12_009
assert len(weak_ctr) == 9_745

print("Checks passed: data grain, public-safe identifiers, and label/leakage exclusions are recorded.")

Checks passed: data grain, public-safe identifiers, and label/leakage exclusions are recorded.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Commit to my repo under `work/notebooks/`, then submit the repo URL on the card.
